# Push HiCWE models to Hugging Face

This notebook publishes the two HiCWE (Hierarchy-Aware Contrastive CWE Encoder) sentence
transformers to the Hugging Face Hub:

| Repo | Base model | Role |
|---|---|---|
| `regularpooria/HiCWE-MiniLM-L6-v2` | `all-MiniLM-L6-v2` | lightweight default |
| `regularpooria/HiCWE-MiniLM-L12-v2` | `all-MiniLM-L12-v2` | larger, best @1 |

Each repo is a **merged** `sentence-transformers` model (Recipe A, *hier full*): the LoRA
adapter trained in `experiments/model_finetuning.ipynb` is merged into the base weights, so
users load it with a plain `SentenceTransformer("regularpooria/HiCWE-...")` and no PEFT/LoRA
runtime.

The LoRA adapters live inside the zip archives produced by the training notebook. This
notebook extracts the `saved_A_hier_full_lora` adapter from each zip, merges it, stages a
complete model folder under `hf_upload/`, then uploads the folders.


### Requirements

`transformers`/`peft`/`sentence-transformers` to merge and smoke-test the models,
`huggingface_hub` for the upload, and `python-dotenv` to load `HF_TOKEN` from the project
`.env` file.


In [ ]:
!pip install -q transformers peft sentence-transformers huggingface_hub python-dotenv

## Configuration

- `BASE_DIR` points at the repository root. It is `"../"` when this notebook runs from
  `models/`. In Colab/Kaggle, uncomment the clone lines below and adjust if needed.
- `MODELS` maps each Hub repo id to its adapter zip and the base model used during training.
- `MODELS[...]["zip"]` are the archives from the training run. They are **not** committed to
  git, so in Colab/Kaggle you must upload them next to this notebook (or set `ZIP_DIR` to
  wherever they live).
- `HF_TOKEN` is read from `.env` at the repository root (`HF_TOKEN=hf_...`). The token needs
  write access to the repositories.


In [ ]:
from pathlib import Path

# == KAGGLE/COLAB ==
# !git clone https://github.com/regularpooria/CWE_CVE_Mapping
# %cd CWE_CVE_Mapping/models
# BASE_DIR = "../"

# == LOCAL (run this notebook from the models/ folder) ==
BASE_DIR = Path("../")

# Where the training zips are. They are gitignored, so upload them in Colab/Kaggle
# or place them at this path locally.
ZIP_DIR = BASE_DIR / "experiments"

# Hub repo id -> (adapter zip, base model, card file)
MODELS = {
    "regularpooria/HiCWE-MiniLM-L6-v2": {
        "zip": ZIP_DIR / "minilm l6.zip",
        "base": "unsloth/all-MiniLM-L6-v2",
        "card": "model_cards/HiCWE-MiniLM-L6-v2.md",
    },
    "regularpooria/HiCWE-MiniLM-L12-v2": {
        "zip": ZIP_DIR / "minilm l12.zip",
        "base": "sentence-transformers/all-MiniLM-L12-v2",
        "card": "model_cards/HiCWE-MiniLM-L12-v2.md",
    },
}

# Folder name inside the zip that holds the merged recipe.
ADAPTER_SUBDIR = "saved_A_hier_full_lora"

UPLOAD_ROOT = Path("hf_upload")
LICENSE_SRC = Path("model_cards/LICENSE")

for repo_id, spec in MODELS.items():
    print(repo_id, "->", spec["zip"], "(exists:", spec["zip"].exists(), ")")

## Extract and merge the LoRA adapter

The zip stores a `sentence-transformers` folder plus an un-merged LoRA adapter
(`adapter_model.safetensors`). We copy the folder, swap the transformer weights for the
merged ones, and drop the `adapter_*` files so the result is a standalone encoder.


In [ ]:
import glob
import json
import shutil
import tempfile
import zipfile
from pathlib import Path

from peft import PeftModel
from transformers import AutoModel


def extract_adapter(zip_path: Path, dest: Path) -> Path:
    """Extract just the ADAPTER_SUBDIR folder from the training zip."""
    with zipfile.ZipFile(zip_path) as z:
        names = z.namelist()
        cfg = next(
            (n for n in names if n.endswith(f"{ADAPTER_SUBDIR}/adapter_config.json")), None
        )
        if cfg is None:
            raise FileNotFoundError(
                f"{ADAPTER_SUBDIR}/adapter_config.json not found in {zip_path}"
            )
        subdir = cfg[: -len("/adapter_config.json")]
        members = [n for n in names if n.startswith(subdir + "/")]
        z.extractall(dest, members)
    return dest / subdir


def merge_lora(src: Path, dst: Path) -> None:
    """Merge the LoRA adapter at src into its base model and write a full model to dst."""
    base_id = json.load(open(src / "adapter_config.json"))["base_model_name_or_path"]
    print(f"  base model: {base_id}")

    base = AutoModel.from_pretrained(base_id).float()
    merged = PeftModel.from_pretrained(base, str(src)).merge_and_unload()

    # Keep the whole sentence-transformers folder (modules.json, pooling, normalize,
    # tokenizer, max_seq_length); only the transformer weights are replaced.
    shutil.copytree(src, dst, dirs_exist_ok=True)
    for f in glob.glob(f"{dst}/adapter_*"):
        Path(f).unlink()
    merged.save_pretrained(dst)


def build_model(repo_id: str, spec: dict) -> Path:
    if not spec["zip"].exists():
        raise FileNotFoundError(
            f"Missing {spec['zip']}. Upload the training zips or fix ZIP_DIR."
        )
    out_dir = UPLOAD_ROOT / repo_id.split("/", 1)[1]
    print(f"Building {out_dir} ...")
    with tempfile.TemporaryDirectory() as tmp:
        adapter = extract_adapter(spec["zip"], Path(tmp))
        merge_lora(adapter, out_dir)
    print(f"  wrote {out_dir}")
    return out_dir


STAGED = {repo_id: build_model(repo_id, spec) for repo_id, spec in MODELS.items()}

## Stage the model cards and license

Copy the edited cards from `model_cards/` into each staged folder as `README.md`, and drop in
the Apache-2.0 `LICENSE`. Edit the cards in `model_cards/` (not the staged copy) so a re-run
picks up your changes.


In [ ]:
for repo_id, out_dir in STAGED.items():
    card = Path(MODELS[repo_id]["card"])
    shutil.copyfile(card, out_dir / "README.md")
    shutil.copyfile(LICENSE_SRC, out_dir / "LICENSE")
    print(f"{out_dir}: README.md <- {card.name}, LICENSE")

for out_dir in STAGED.values():
    print(out_dir)
    for f in sorted(out_dir.iterdir()):
        print("   ", f.name)

## Smoke test

Load each merged model as a plain `SentenceTransformer` and check that a CVE description
scores its matching CWE text higher than an unrelated one.


In [ ]:
from sentence_transformers import SentenceTransformer, util

cve = "The product does not neutralize special elements used in an SQL command."
related = (
    "Improper Neutralization of Special Elements used in an SQL Command ('SQL Injection'). "
    "The product constructs all or part of an SQL command using externally-influenced input."
)
unrelated = (
    "Improper Restriction of Operations within the Bounds of a Memory Buffer. "
    "The product performs operations on a memory buffer, but it can read from or write to "
    "a memory location that is outside of the intended boundary."
)

for repo_id, out_dir in STAGED.items():
    model = SentenceTransformer(str(out_dir))
    emb = model.encode([cve, related, unrelated], normalize_embeddings=True)
    sim_related = float(util.cos_sim(emb[0], emb[1]))
    sim_unrelated = float(util.cos_sim(emb[0], emb[2]))
    print(f"{repo_id}: related={sim_related:.4f}  unrelated={sim_unrelated:.4f}  "
          f"{'OK' if sim_related > sim_unrelated else 'CHECK'}")

## Push to the Hub

Create the repos if they do not exist and commit the staged folders. `repo_type="model"`.


In [ ]:
import os

from dotenv import load_dotenv
from huggingface_hub import HfApi

load_dotenv(BASE_DIR / ".env")
api = HfApi(token=os.environ.get("HF_TOKEN"))

for repo_id, out_dir in STAGED.items():
    api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
    url = api.upload_folder(
        folder_path=str(out_dir),
        repo_id=repo_id,
        repo_type="model",
        commit_message="Upload HiCWE model",
    )
    print(f"{repo_id}: {url}")

## Add the models to the collection

The models are grouped in the existing Hub collection
`https://huggingface.co/collections/regularpooria/cve-cwe`. This is the link to cite in the
paper. Re-running the cell is safe: items already present are skipped.


In [ ]:
from huggingface_hub import add_collection_item, get_collection

# The existing collection the two repos belong to (this is the paper's link).
COLLECTION_SLUG = "regularpooria/cve-cwe"

collection = get_collection(COLLECTION_SLUG, token=os.environ.get("HF_TOKEN"))
existing = {item.item_id for item in collection.items}

for repo_id in MODELS:
    if repo_id in existing:
        print(f"already in collection: {repo_id}")
        continue
    add_collection_item(
        collection_slug=COLLECTION_SLUG,
        item_id=repo_id,
        item_type="model",
        exists_ok=True,
    )
    print(f"added: {repo_id}")

print(f"collection: https://huggingface.co/collections/{COLLECTION_SLUG}")

## Notes

- Re-running overwrites the staged folders and re-commits, so this is safe to run after any
  change to the cards or the adapters.
- The merged models are ~90 MB (L6) and ~180 MB (L12). If you also want the raw adapters on
  the Hub, upload `saved_A_hier_full_lora` into a `lora/` subfolder and mention it in the card.
- If you rename the repos, update `MODELS`, the `base_model` field in the cards, and the
  collection cell.
